In [1]:
from sage.all import*

In [2]:
def Pivot(A, i, W):
    m = A.ncols()
    I = -1
    D = -1
    DW = -1  # shifted pivot degree

    for j in range(m):
        if A[i,j] != 0:
            if A[i,j].degree() + W[j] >= DW: 
                DW = A[i,j].degree()+W[j]
                D = A[i,j].degree()
                I = j

    return (I, D, DW)

def SimpleTransformation(A,i,j,I,d_i,d_j):
    m = A.ncols()
    d = d_i-d_j

    cxe = (A[i,I].leading_coefficient() / A[j,I].leading_coefficient())*(x**d)
    for l in range(m):
        A[i,l] = A[i,l] - cxe*A[j,l]

    return A

In [3]:
def WeakPopov(A,W=0):
    n = A.nrows()
    # If no weights are given, set W = [0,0,...,0]
    if W == 0:
        W=zero_vector(ZZ, A.nrows())
        
    pivots = zero_matrix(ZZ, n, 2)

    control = 1

    for i in range(n):
        pivots[i,0] = Pivot(A,i,W)[0] 
        pivots[i,1] = Pivot(A,i,W)[1]

    while control == 1:

        control = 0
        broke = False

        for i in range(n):
            for j in range(i+1,n):
                if pivots[i,0] == pivots[j,0]:
                    if pivots[i,1] >= pivots[j,1]:
                        A = SimpleTransformation(A,i,j,pivots[i,0],pivots[i,1],pivots[j,1])
                        pivots[i,0] = Pivot(A,i,W)[0]
                        pivots[i,1] = Pivot(A,i,W)[1]
                    else:
                        A = SimpleTransformation(A,j,i,pivots[j,0],pivots[j,1],pivots[i,1])
                        pivots[j,0] = Pivot(A,j,W)[0]
                        pivots[j,1] = Pivot(A,j,W)[1]
                    control = 1
                    broke = True
                    break
            if broke:
                break
                        

    return A
    

In [4]:
def MakeMonic(A,pivots,n,m):

    for i in range(n):
        if A[i,pivots[i,0]].leading_coefficient() != 1 and A[i,pivots[i,0]].leading_coefficient() != 0:
            monic_const = 1 / A[i,pivots[i,0]].leading_coefficient() 
            for j in range(m):
                A[i,j] = monic_const*A[i,j]  
    return A 

def AscendingOrder(A, W):
    n = A.nrows()
    check = 1
    while check == 1:
        check = 0
        for i in range(n-1):
            I1, D1, DW1 = Pivot(A, i, W)
            I2, D2, DW2 = Pivot(A, i+1, W)

            if DW1 > DW2:
                A.swap_rows(i, i+1)
                check = 1
            elif DW1 == DW2:
                if I1 > I2:
                    A.swap_rows(i, i+1)
                    check = 1
    return A

In [5]:
def PopovForm(A,W=0):
    n = A.nrows()
    # If no weights are given, set W = [0,0,...,0]
    if W == 0:
        W=zero_vector(ZZ, A.nrows())
    m = A.ncols()
    pivots = zero_matrix(ZZ, n, 2)
    C = []

    A = WeakPopov(A,W)
    A = AscendingOrder(A,W)

    for i in range(n):
        pivots[i,0] = Pivot(A,i,W)[0]
        pivots[i,1] = Pivot(A,i,W)[1]
    
    for k in range(n):
        if not A[k].is_zero():
            C.append(k)
            for i in C:
                if i<k:
                    delta = A[k,pivots[i,0]].degree() - pivots[i,1]
                    if delta >= 0:
                        A = SimpleTransformation(A,k,i,pivots[i,0],A[k,pivots[i,0]].degree(),pivots[i,1])
    A = MakeMonic(A,pivots,n,m)  
      
    return A

In [6]:
def G_Poly(eval,R):
    x = R.gen()
    G = 1
    for xi in eval:
        G *= (x-xi)
    return G

def LagrangePolynomials(eval, R, n):
    x = R.gen()
    L = vector(R, [1]*n)
    for i in range(n):
        for l in range(n):
            if l == i:
                continue
            L[i] *= (x-eval[l])/(eval[i]-eval[l])
    return L

def R_Poly(L,r,n):
    RX = 0
    for i in range(n):
        RX += r[i]*L[i]

    return RX

In [7]:
def SudanListDecoding(eval,r,l,R,k):
    n = eval.length()
    G = G_Poly(eval,R)
    L = LagrangePolynomials(eval, R, n)
    RX = R_Poly(L,r,n)
    M = matrix.identity(R, l+1, l+1)
    M[0,0] = G
    for i in range(1,l+1):
        M[i,0] = -RX**i
    W = vector(ZZ, range(l+2))
    M = WeakPopov(M,W)
    # Choose row with lowest weightet degree
    best_row = None
    best_degree = None
    for i in range(l+1):
        rowdegree = None
        for j in range(l+1):
            if rowdegree is None:
                rowdegree = M[i,j].degree()+W[j]
            else:
                rowdegree = max(M[i,j].degree()+W[j], rowdegree)
        if best_degree is None or best_degree >= rowdegree:
            best_row = i
            best_degree = rowdegree
    Q = 0
    for i in range(l+1):
        Q += M[best_row][i]*y**i
    print(Q)
    
    # Check that f agrees with enough points

    candidates = []
    valid_candidates = []
    tau = floor(n*l/(l+1)-l/2*(k-1))
    threshold = n - tau

    for factor, _ in Q.factor():
        if factor.degree(y) == 1:
            a = factor.coefficient(y) 
            b = factor - a*y 

            f = -b / a      
            candidates.append(f)

            agreements = 0
            for xi, ri in zip(eval, r):
                if f(xi,y) == ri:
                    agreements += 1
            if agreements >= threshold:
                valid_candidates.append(f)
    return candidates, valid_candidates    

In [8]:
def GuruswamiSudanListDecoding(eval,r,l,q,k,m=1):
    n = eval.length()

    F = GF(q)
    R = PolynomialRing(F, "x,y")

    # CHANGED: make a univariate ring S = F[x] for the Popov matrix
    F = R.base_ring()
    S = PolynomialRing(F, "x")
    xS = S.gen()

    # CHANGED: keep bivariate variables only for final Q and factorization
    x, y = R.gens()

    # CHANGED: compute G and RX in S, not in R = F[x,y]
    G = prod(xS - S(xi) for xi in eval)
    L = LagrangePolynomials(eval, S, n)
    RX = R_Poly(L, r, n)

    # CHANGED: build matrix over S = F[x]
    M = zero_matrix(S, l+1, l+1)

    # CHANGED: standard multiplicity basis:
    # for i <= m:  G^(m-i)(y-RX)^i
    # for i >  m:  y^(i-m)(y-RX)^m
    for i in range(l+1):
        if i <= m:
            for j in range(i+1):
                M[i,j] = binomial(i,j) * (-RX)**(i-j) * G**(m-i)
        else:
            for j in range(m+1):
                M[i,j+i-m] = binomial(m,j) * (-RX)**(m-j)

    # CHANGED: weighted degree should be x-degree + j*(k-1)
    W = vector(ZZ, [j*(k-1) for j in range(l+1)])

    M = WeakPopov(M,W) #Can use weakpopov and popov

    # Choose row with lowest weighted degree
    best_row = None
    best_degree = None
    for i in range(l+1):
        rowdegree = None
        for j in range(l+1):
            if M[i,j] == 0:       # CHANGED: skip zero entries
                continue

            deg = M[i,j].degree() + W[j]

            if rowdegree is None:
                rowdegree = deg
            else:
                rowdegree = max(deg, rowdegree)

        if rowdegree is not None and (best_degree is None or best_degree >= rowdegree):
            best_row = i
            best_degree = rowdegree

    # CHANGED: reconstruct Q in the bivariate ring R
    Q = 0
    for i in range(l+1):
        Q += R(M[best_row][i]) * y**i

    print(Q)

    candidates = []
    valid_candidates = []

    # CHANGED: multiplicity decoding condition
    threshold = floor(best_degree/m) + 1

    for factor, _ in Q.factor():
        if factor.degree(y) == 1:
            a = factor.coefficient(y)
            b = factor - a*y

            f = -b / a
            candidates.append(f)

            agreements = 0
            for xi, ri in zip(eval, r):
                if f(xi, y) == ri:
                    agreements += 1

            if agreements >= threshold:
                valid_candidates.append(f)

    return candidates, valid_candidates

In [10]:
k = 2
l = 3
q=13
F = GF(q)

R = PolynomialRing(F, "x,y")
x, y = R.gens()

eval = vector(F, [1,2,3,4,5,6])
r = vector(F, [2,3,4,8,6,3])
n = 6

#floor(n*l/(l+1)-l/2*(k-1))
GuruswamiSudanListDecoding(eval, r, l, q, k,2)

2*x^5 + 6*x^4*y - 2*x^3*y^2 - 6*x^2*y^3 + 4*x^4 + 6*x^3*y + 3*x^2*y^2 + 3*x*y^3 + x^3 + 3*x^2*y - 5*y^3 - 4*x^2 - 2*x*y - y^2 - 2*x + 5*y + 1


([4*x + 5, x + 1, (-x^3 - 3*x^2 + x - 3)/(x^2 + 6*x + 3)],
 [4*x + 5, x + 1, (-x^3 - 3*x^2 + x - 3)/(x^2 + 6*x + 3)])